# Telecom Revenue & Customer Behavior Analytics

This clean, reproducible notebook loads the local Megaline tables, creates one row per observed customer-month, applies plan billing rules, and tests revenue differences by plan and region. Raw records are not embedded in this notebook.

In [1]:
from pathlib import Path
import sys

import pandas as pd

sys.path.insert(0, str(Path.cwd()))
from src.megaline_analytics import analyze, welch_t_test

data, customer_month = analyze("data")
print({name: frame.shape for name, frame in data.items()})
print("Customer-month dimensions:", customer_month.shape)

{'users': (500, 8), 'calls': (137735, 4), 'messages': (76051, 3), 'internet': (104825, 4), 'plans': (2, 8)}
Customer-month dimensions: (2293, 23)


## Monthly usage and billing

The billing model rounds each call up before aggregation, rounds total monthly data to whole GB, and charges only positive overages.

In [2]:
plan_summary = customer_month.groupby("plan")["monthly_revenue"].agg(["count", "mean", "median", "std"])
plan_summary

,count,mean,median,std
plan,,,,
surf,1573,60.706408,40.36,55.388042
ultimate,720,72.313889,70.00,11.395108


In [3]:
surf = customer_month.loc[customer_month["plan"] == "surf", "monthly_revenue"]
ultimate = customer_month.loc[customer_month["plan"] == "ultimate", "monthly_revenue"]
plan_t, plan_p = welch_t_test(surf, ultimate)
print(f"Welch t-test, Surf vs Ultimate: t={plan_t:.4f}, p={plan_p:.6g}")

Welch t-test, Surf vs Ultimate: t=-7.9521, p=3.17039e-15


In [4]:
ny_nj = customer_month.loc[
    customer_month["city"] == "New York-Newark-Jersey City, NY-NJ-PA MSA",
    "monthly_revenue",
]
other_regions = customer_month.loc[
    customer_month["city"] != "New York-Newark-Jersey City, NY-NJ-PA MSA",
    "monthly_revenue",
]
region_t, region_p = welch_t_test(ny_nj, other_regions)
print(f"Welch t-test, NY-NJ vs other regions: t={region_t:.4f}, p={region_p:.6g}")

Welch t-test, NY-NJ vs other regions: t=-2.1309, p=0.0335256


## Interpretation

The descriptive tables and p-values above are the authoritative results for this local dataset. A small p-value indicates evidence of a difference in average observed monthly revenue; it does not identify a causal effect or guarantee future performance.